# 12 · Fusión en español: DistilBERT multilingüe (E2) + rama URL — Grupo 17 · cierre de la vía B
**Entorno CPU.** Usa probabilidades ya calculadas (`text_scores_multi_es_v1.parquet`, del 11/E2) y la rama URL existente
(`url_xgb_v1b_correos.json`, entrenada solo con URLs de correos en inglés: en español es transferencia pura).

Métricas **binarias** (phishing vs legítimo): no hay spam nativo en español y un F1 macro de 3 clases lo penalizaría artificialmente.
El meta-clasificador se ajusta en el **val nativo** y se evalúa en el **test nativo** (nunca vistos por la rama de texto en su entrenamiento;
el val sí se usó para early stopping en E2: leve optimismo, declarado). Política de umbral: FPR ≤ 1 % en validación (D9).

| Variante | Entradas |
|---|---|
| V1_es | DistilBERT multilingüe |
| V2_es | DistilBERT multilingüe + rama URL |

In [1]:
# Celda 1: Entorno y entradas
import os, sys, json, time, random, importlib.util
from pathlib import Path
import numpy as np, pandas as pd
SEED = 17; random.seed(SEED); np.random.seed(SEED)
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
PROC, MODELS, RES = BASE / "data/processed", BASE / "models", BASE / "results"
def cargar(nombre, ruta):
    spec = importlib.util.spec_from_file_location(nombre, ruta); m = importlib.util.module_from_spec(spec)
    sys.modules[nombre] = m; spec.loader.exec_module(m); return m
ioc = cargar("ioc_utils", BASE / "src/ioc_utils.py"); uf = cargar("url_features", BASE / "src/url_features.py")
ES = pd.read_parquet(PROC / "emails_es_v2.parquet", columns=["id", "grupo", "clase", "y", "split", "urls_defang"])
ES = ES[ES["grupo"].isin(["SpaPhish", "SpearPhishMX"]) & ES["split"].isin(["val", "test"])].copy()   # solo nativo
T = pd.read_parquet(PROC / "text_scores_multi_es_v1.parquet")
D = ES.merge(T[["id", "p_legitimo", "p_spam", "p_phishing"]], on="id", how="inner")
assert len(D) == len(ES), "Las probabilidades del 11/E2 no cubren el val+test nativo"
D["yb"] = (D["clase"] == "phishing").astype(int)
print(ENV, "| val+test nativo:", len(D), D.groupby(["split", "clase"]).size().to_dict())

Mounted at /content/drive
colab | val+test nativo: 1281 {('test', 'legitimo'): 254, ('test', 'phishing'): 387, ('val', 'legitimo'): 254, ('val', 'phishing'): 386}


In [2]:
# Celda 2: Rama URL sobre el español nativo (transferencia pura)
import xgboost as xgb
from sklearn.metrics import roc_auc_score
mu = xgb.XGBClassifier(); mu.load_model(MODELS / "url_xgb_v1b_correos.json")
TLD = json.loads((MODELS / "url_xgb_v1.meta.json").read_text())["tld_categorias"]
filas = [(i, ioc.refang(u)) for i, l in zip(D["id"], D["urls_defang"]) for u in list(l)[:20]]
U = pd.DataFrame(filas, columns=["id", "url"])                      # vivas SOLO en memoria
if len(U):
    F = pd.DataFrame([uf.features_url(u) for u in U["url"]]); U = U.drop(columns="url")
    X = F[uf.NUMERICAS].astype(float).copy()
    X["tld"] = pd.Categorical(F["tld"].where(F["tld"].isin(TLD), "__otro__"), categories=TLD)
    U["s"] = mu.predict_proba(X)[:, 1]
    agg = U.groupby("id")["s"].agg(url_max="max", url_media="mean").join(U.groupby("id").size().rename("n_urls"))
else:
    agg = pd.DataFrame(columns=["url_max", "url_media", "n_urls"])
D = D.merge(agg.reset_index(), on="id", how="left"); D["n_urls"] = D["n_urls"].fillna(0)
def auc(y, s):
    m = ~pd.isna(s); return round(float(roc_auc_score(y[m], s[m])), 4) if m.sum() and len(set(y[m])) > 1 else None
URL_SOLA = {g: auc(sub["yb"].values, sub["url_max"].values) for g, sub in D[D.split == "test"].groupby("grupo")}
URL_SOLA["pct_con_url"] = {g: round(100 * float((sub["n_urls"] > 0).mean()), 1) for g, sub in D.groupby("grupo")}
URL_SOLA

{'SpaPhish': 0.374,
 'SpearPhishMX': 0.4451,
 'pct_con_url': {'SpaPhish': 80.9, 'SpearPhishMX': 65.4}}

In [3]:
# Celda 3: Meta binario calibrado, umbral FPR≤1 %, variantes
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
lg = lambda p: np.log(np.clip(p, 1e-6, 1.0))
def feats(v, df):
    base = lg(df[["p_legitimo", "p_spam", "p_phishing"]].values)
    if v == "V1_es": return base
    u = np.c_[df["url_max"].fillna(0.5), df["url_media"].fillna(0.5), (df["n_urls"] == 0).astype(float), np.log1p(df["n_urls"])]
    return np.c_[base, u]
def umbral_fpr(y, p, fpr=0.01):
    neg = np.sort(p[y == 0])[::-1]; k = int(np.floor(fpr * len(neg)))
    return float(np.nextafter(neg[k], 1)) if k < len(neg) else 0.0
def brier(y, p): return round(float(np.mean((p - y) ** 2)), 5)
def ece(y, p, b=10):
    e = 0.0
    for lo in np.linspace(0, 1, b, endpoint=False):
        m = (p >= lo) & ((p < lo + 1 / b) if lo < 1 - 1 / b else True)
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return round(float(e), 5)
def binario(y, pred):
    tp, fp = int(((pred == 1) & (y == 1)).sum()), int(((pred == 1) & (y == 0)).sum())
    fn, tn = int(((pred == 0) & (y == 1)).sum()), int(((pred == 0) & (y == 0)).sum())
    return {"recall_phishing": round(tp / max(tp + fn, 1), 4), "precision_phishing": round(tp / max(tp + fp, 1), 4),
            "fpr_legitimo_a_phishing": round(fp / max(fp + tn, 1), 4), "f1_phishing": round(f1_score(y, pred, zero_division=0), 4),
            "matriz_[tn,fp,fn,tp]": [tn, fp, fn, tp], "n": int(len(y))}
va, te = (D["split"] == "val").values, (D["split"] == "test").values
yv, yt = D.loc[va, "yb"].values, D.loc[te, "yb"].values
RES_F, MOD = {}, {}
for v in ("V1_es", "V2_es"):
    Xv, Xt = feats(v, D[va]), feats(v, D[te]); cv = StratifiedKFold(5, shuffle=True, random_state=SEED); mejor = None
    for metodo in ("sigmoid", "isotonic"):
        pcv = cross_val_predict(CalibratedClassifierCV(LogisticRegression(max_iter=3000), method=metodo, cv=5), Xv, yv, cv=cv, method="predict_proba")[:, 1]
        b = brier(yv, pcv)
        if mejor is None or b < mejor[1]: mejor = (metodo, b, pcv)
    metodo, bv, pcv = mejor; u = umbral_fpr(yv, pcv)
    m = CalibratedClassifierCV(LogisticRegression(max_iter=3000), method=metodo, cv=5).fit(Xv, yv); pt = m.predict_proba(Xt)[:, 1]
    RES_F[v] = {"calibracion": metodo, "brier_val_cv": bv, "umbral_fpr1": round(u, 5),
                "val_cv": binario(yv, (pcv >= u).astype(int)), "test": binario(yt, (pt >= u).astype(int)),
                "auc_test": auc(yt, pt), "brier_test": brier(yt, pt), "ece_test": ece(yt, pt),
                "test_por_fuente": {g: binario(yt[(D.loc[te, "grupo"] == g).values], (pt >= u).astype(int)[(D.loc[te, "grupo"] == g).values])
                                    for g in ("SpaPhish", "SpearPhishMX")}}
    MOD[v] = (m, u)
    print(v, metodo, "| test recall", RES_F[v]["test"]["recall_phishing"], "FPR", RES_F[v]["test"]["fpr_legitimo_a_phishing"], "AUC", RES_F[v]["auc_test"])

V1_es sigmoid | test recall 0.9354 FPR 0.0354 AUC 0.9842
V2_es sigmoid | test recall 0.9302 FPR 0.0394 AUC 0.9882


In [4]:
# Celda 4: Selección, guardado y RESUMEN
import joblib
# Regla D21: si la rama URL sola no supera el azar (AUC ≤ 0,5) en NINGUNA fuente nativa, no transfiere al español y se excluye.
aucs_url = [v for k, v in URL_SOLA.items() if k != "pct_con_url" and v is not None]
URL_TRANSFIERE = any(a > 0.5 for a in aucs_url)
if not URL_TRANSFIERE:
    ELEGIDA = "V1_es"; MOTIVO = f"regla D21: rama URL sola por debajo del azar en todas las fuentes ({aucs_url})"
else:
    ELEGIDA = max(RES_F, key=lambda v: (RES_F[v]["val_cv"]["recall_phishing"], -RES_F[v]["brier_val_cv"]))
    MOTIVO = "mayor recall en val-CV con FPR≤1 %, desempate por Brier"
for v, (mv, uv) in MOD.items():   # se guardan ambas variantes para trazabilidad
    joblib.dump({"modelo": mv, "umbral_fpr1": uv, "variante": v, "binario": True}, MODELS / f"fusion_es_{v}.joblib")
m, u = MOD[ELEGIDA]
joblib.dump({"modelo": m, "umbral_fpr1": u, "variante": ELEGIDA, "binario": True}, MODELS / "fusion_es_v1.joblib")
(MODELS / "fusion_es_v1.meta.json").write_text(json.dumps({"variante": ELEGIDA, "umbral_phishing": u, "calibracion": RES_F[ELEGIDA]["calibracion"],
    "texto": "distilbert_multi_v1 (11/E2)", "features": "log(P multi)[3]" + (" + url_max, url_media (NaN->0.5), sin_url, log1p(n)" if ELEGIDA == "V2_es" else ""),
    "salida": "P(phishing) binaria calibrada; legítimo vs spam por argmax del texto"}, indent=1, ensure_ascii=False))
RESUMEN = {"entorno": ENV, "url_sola_auc_test_por_fuente": URL_SOLA, "url_transfiere": URL_TRANSFIERE,
           "variantes": RES_F, "elegida": ELEGIDA, "motivo": MOTIVO}
(RES / "fusion_es_v1.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (12) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (12) =====
{"entorno":"colab","url_sola_auc_test_por_fuente":{"SpaPhish":0.374,"SpearPhishMX":0.4451,"pct_con_url":{"SpaPhish":80.9,"SpearPhishMX":65.4}},"url_transfiere":false,"variantes":{"V1_es":{"calibracion":"sigmoid","brier_val_cv":0.03725,"umbral_fpr1":0.84689,"val_cv":{"recall_phishing":0.9223,"precision_phishing":0.9944,"fpr_legitimo_a_phishing":0.0079,"f1_phishing":0.957,"matriz_[tn,fp,fn,tp]":[252,2,30,356],"n":640},"test":{"recall_phishing":0.9354,"precision_phishing":0.9757,"fpr_legitimo_a_phishing":0.0354,"f1_phishing":0.9551,"matriz_[tn,fp,fn,tp]":[245,9,25,362],"n":641},"auc_test":0.9842,"brier_test":0.04416,"ece_test":0.02233,"test_por_fuente":{"SpaPhish":{"recall_phishing":0.8972,"precision_phishing":0.9143,"fpr_legitimo_a_phishing":0.0947,"f1_phishing":0.9057,"matriz_[tn,fp,fn,tp]":[86,9,11,96],"n":202},"SpearPhishMX":{"recall_phishing":0.95,"precision_phishing":1.0,"fpr_legitimo_a_phishing":0.0,"f1_phishing":0.9744,"matriz_[tn,fp,fn,tp]":[159,0,14,266]